# PCOS and Ultrasound Image Analysis Model Training

This notebook trains two models:
1. A neural network for PCOS prediction based on patient data
2. A CNN for analyzing ultrasound images

The models will be combined and converted to TFLite format for use in the Flutter app.

## 1. Import Required Libraries

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
import os
import cv2

## 2. Load and Preprocess PCOS Dataset

In [ ]:
# Load PCOS dataset
pcos_data = pd.read_csv('PCOS_data.csv')

# Select relevant features
features = ['Age', 'BMI', 'Cycle_length', 'Weight_gain', 
           'hair_growth', 'Skin_darkening', 'Hair_loss', 'Pimples', 
           'Fast_food', 'Reg_Exercise', 'BP_Systolic', 'BP_Diastolic',
           'Follicle_No_R', 'Follicle_No_L', 'Avg_F_size_R', 'Avg_F_size_L']

X = pcos_data[features]
y = pcos_data['PCOS']

# Handle missing values
X = X.fillna(X.mean())

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

## 3. Build and Train the PCOS Prediction Model

In [ ]:
def create_pcos_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Dense(64, activation='relu')(inputs)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.2)(x)
    outputs = Dense(3, activation='softmax')(x)  # 3 classes: Low, Moderate, High risk
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Create and compile model
pcos_model = create_pcos_model((len(features),))
pcos_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train model
history = pcos_model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

# Evaluate model
test_loss, test_accuracy = pcos_model.evaluate(X_test, y_test)
print(f'Test accuracy: {test_accuracy:.4f}')

## 4. Convert PCOS Model to TFLite Format

In [ ]:
# Convert PCOS model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(pcos_model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,
                                     tf.lite.OpsSet.SELECT_TF_OPS]
converter.optimizations = [tf.lite.Optimize.DEFAULT]
pcos_tflite_model = converter.convert()

# Save PCOS TFLite model
with open('pcos_model_temp.tflite', 'wb') as f:
    f.write(pcos_tflite_model)

## 5. Load and Preprocess Ultrasound Image Dataset

In [ ]:
# Set up image data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Load and preprocess images
train_generator = train_datagen.flow_from_directory(
    'ultrasound_images',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    'ultrasound_images',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

## 6. Build and Train the CNN Model for Ultrasound Analysis

In [ ]:
def create_cnn_model():
    base_model = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze the base model
    base_model.trainable = False
    
    # Add custom layers
    x = Flatten()(base_model.output)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=base_model.input, outputs=outputs)
    return model

# Create and compile CNN model
cnn_model = create_cnn_model()
cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train CNN model
history_cnn = cnn_model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator
)

## 7. Combine Models into a Single TFLite Model

In [ ]:
# Convert CNN model to TFLite
converter_cnn = tf.lite.TFLiteConverter.from_keras_model(cnn_model)
converter_cnn.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,
                                         tf.lite.OpsSet.SELECT_TF_OPS]
converter_cnn.optimizations = [tf.lite.Optimize.DEFAULT]
cnn_tflite_model = converter_cnn.convert()

# Create a combined model
class CombinedModel(tf.keras.Model):
    def __init__(self, pcos_model, cnn_model):
        super(CombinedModel, self).__init__()
        self.pcos_model = pcos_model
        self.cnn_model = cnn_model
    
    def call(self, inputs):
        pcos_pred = self.pcos_model(inputs[0])
        cnn_pred = self.cnn_model(inputs[1])
        return [pcos_pred, cnn_pred]

combined_model = CombinedModel(pcos_model, cnn_model)

# Convert and save the final combined model
converter_combined = tf.lite.TFLiteConverter.from_keras_model(combined_model)
converter_combined.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,
                                              tf.lite.OpsSet.SELECT_TF_OPS]
converter_combined.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_combined_model = converter_combined.convert()

# Save the final model
with open('../assets/models/pcos_model.tflite', 'wb') as f:
    f.write(tflite_combined_model)

print('Models successfully trained and combined into TFLite format!')